# Experiment 0 — Efficiency (T08)

**What it measures:** the cost of each upsampler at inference: parameters, size, run time and peak GPU memory. Forward pass only, under `no_grad` (as in Exp 1–3).
**Upsamplers:** `bilinear`, `anyup2d` (one call per frame), `anyup3d` (one call per chunk).

- **Inputs:** synthetic, fixed seed. Features `(T, 768, 14, 14)` (VideoMAE grid), guide `(T, 3, S, S)` with S = output size. The cost doesn't depend on content, so no dataset or backbone is needed.
- **Main row:** the Exp 2 setting (224², 8 feature frames, AnyUp3D `q_chunk_size=4096`), measured in both seeding modes (`fast` = `deterministic=False`, normal inference; `deterministic` = as in Exp 1–3).
- **Sweep** (`fast` mode): output size × feature frames × AnyUp3D `q_chunk_size`. Rows that run out of memory are recorded as `OOM`.
- **Timing:** CUDA events, warm-up then repeated runs, median and IQR. **Memory:** total peak and extra peak (peak minus memory in use just before the call).

Ratios are AnyUp3D / AnyUp2D, shown next to the thesis claims (+75.5 % params, ≈ 2.7× slower, ≈ 10.5× memory), whose settings are unknown (see `docs/plan/P1_experiments/T08_exp0_efficiency.md`). Results go to `<OUTPUT_ROOT>/<RUN_NAME>/`. **Runtime → Restart session** before running if `anyup` was imported before.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS (T07); must be set before anything starts CUDA
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-task02"   # T08 continues on the T02 branch
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
USE_T4 = False   # ← True = T4 run; False = A100. Sizes / frames / q values are the same on both GPUs (comparable rows)

GPU = "T4" if USE_T4 else "A100"
GPU_PROFILES = {
    "A100": dict(WARMUP=5, REPEATS=20),
    "T4":   dict(WARMUP=2, REPEATS=5),
}
P = GPU_PROFILES[GPU]
WARMUP  = P["WARMUP"]    # A100: 5, T4: 2 — untimed calls before measuring (cuDNN / allocator warm-up); run time only
REPEATS = P["REPEATS"]   # A100: 20, T4: 5 — timed calls per row → median / IQR; run time only, no effect on memory

CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # checkpoint choice skipped for now (S06)
OUTPUT_ROOT  = "/content/drive/MyDrive/results/T08/exp0"
DEVICE       = "cuda"
SEED         = 0

UPSAMPLERS = ["bilinear", "anyup2d", "anyup3d"]
FEAT_DIM, FEAT_GRID = 768, 14    # VideoMAE width and 14² grid (224 / 16); every upsampler's input

MAIN = dict(out_size=224, frames=8, q_chunk=4096)   # ↓ Exp 2 setting: 224² output, 8 feature frames (one 16-frame chunk), q 4096.
                                                    #   Measured in both seeding modes (Q9)
RUN_SWEEP = True
OUT_SIZES = [56, 112, 224]       # ↓ guide = output size → memory/time of every upsampler (AnyUp3D attention ∝ frames·S²). Dependent: Q_CHUNKS
FRAMES    = [4, 8, 16]           # ↓ feature frames per call → AnyUp3D memory/time; AnyUp2D/bilinear scale linearly. Dependent: Q_CHUNKS
Q_CHUNKS  = [None, 1024, 4096]   # ↓ AnyUp3D query chunk → peak memory (None = all queries at once, OOMs at large S·frames). Dependent: OUT_SIZES, FRAMES

THESIS = dict(params_increase_pct=75.5, time_ratio=2.7, memory_ratio=10.5)   # thesis claims vs AnyUp2D (settings unknown, T08 finding B)
ANYUP3D_PARAMS_THESIS = 1_540_736   # sanity check (T08 finding A)
RUN_VISUALS = False                 # optional plot (Cell 7)

RUN_NAME = f"{GPU}_main_s{MAIN['out_size']}_f{MAIN['frames']}_q{MAIN['q_chunk']}" + ("_sweep" if RUN_SWEEP else "")
OUT_DIR  = f"{OUTPUT_ROOT}/{RUN_NAME}"
print(f"GPU={GPU} RUN_NAME={RUN_NAME} WARMUP={WARMUP} REPEATS={REPEATS}")

In [ ]:
# ── Cell 3 · Imports, upsamplers, parameters ───────────────────────────────
import gc, time
import numpy as np
import torch
import matplotlib.pyplot as plt

from anyup import metrics
from anyup.backbones import BackboneOutput
from anyup.upsamplers import UpsamplerLoader
from anyup.utils.seed import set_seed

def set_mode(mode):
    # "fast" = deterministic=False (normal inference); "deterministic" = as Exp 1–3. Seeds are set in both.
    set_seed(SEED, deterministic=(mode == "deterministic"))
    if mode == "fast":                                     # set_seed(False) doesn't undo the deterministic flags
        torch.use_deterministic_algorithms(False)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = False

CHECKS = []
def check(name, passed, detail):
    CHECKS.append(dict(name=name, passed=bool(passed), detail=detail))
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")

GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
if GPU not in GPU_NAME:
    print(f"WARNING: profile {GPU} but the runtime has {GPU_NAME}")

set_mode("fast")
UPS = {"bilinear": UpsamplerLoader.load("bilinear", device=DEVICE),
       "anyup2d":  UpsamplerLoader.load("anyup2d", device=DEVICE),
       "anyup3d":  UpsamplerLoader.load("anyup3d", ckpt_path=CKPT_ANYUP3D, device=DEVICE,
                                        q_chunk_size=MAIN["q_chunk"])}   # changed per row in Cell 5. Dependent: OUT_SIZES, FRAMES

PARAMS = {}
for name in UPSAMPLERS:
    model = getattr(UPS[name], "model", None)
    n = sum(p.numel() for p in model.parameters()) if model is not None else 0
    PARAMS[name] = dict(params=n, size_mib=n * 4 / 2**20)            # fp32
    print(f"{name:<9} params {n:>10,}  size {n * 4 / 2**20:6.2f} MiB")
check("anyup3d params = thesis", PARAMS["anyup3d"]["params"] == ANYUP3D_PARAMS_THESIS,
      f"{PARAMS['anyup3d']['params']:,} vs {ANYUP3D_PARAMS_THESIS:,}")

In [ ]:
# ── Cell 4 · Measurement ───────────────────────────────────────────────────
def make_inputs(frames, out_size):
    g = torch.Generator(device="cpu").manual_seed(SEED)
    feats = torch.randn(frames, FEAT_DIM, FEAT_GRID, FEAT_GRID, generator=g)   # (T, 768, 14, 14) raw features
    guide = torch.randn(frames, 3, out_size, out_size, generator=g)            # (T, 3, S, S) "normalized" guide
    return BackboneOutput(features=feats.to(DEVICE), guide=guide.to(DEVICE), meta={})

def measure(fn, warmup, repeats):
    # → per-call times (ms), memory in use before the timed calls, peak during them, output shape
    for _ in range(warmup):
        out = fn(); del out
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()                   # inputs + weights (+ allocator state)
    times, shape = [], None
    for _ in range(repeats):
        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        start.record()
        out = fn()
        end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))
        shape = tuple(out.shape)
        del out
    return times, base, torch.cuda.max_memory_allocated(), shape

def run_row(part, mode, name, out_size, frames, q_chunk):
    row = dict(part=part, mode=mode, upsampler=name, out_size=out_size, frames=frames,
               q_chunk=q_chunk if name == "anyup3d" else None, repeats=REPEATS)
    up = UPS[name]
    if name == "anyup3d":
        up.q_chunk_size = q_chunk                          # ↓ AnyUp3D query chunk for this row
    chunk = make_inputs(frames, out_size)
    try:
        times, base, peak, shape = measure(lambda: up(chunk, output_size=out_size).features, WARMUP, REPEATS)
        q1, med, q3 = np.percentile(times, [25, 50, 75])
        row.update(status="ok", median_ms=float(med), iqr_ms=float(q3 - q1), per_frame_ms=float(med / frames),
                   peak_total_mib=peak / 2**20, peak_extra_mib=(peak - base) / 2**20, out_shape=list(shape))
    except torch.cuda.OutOfMemoryError:
        row.update(status="OOM")
    finally:
        del chunk
        gc.collect()
        torch.cuda.empty_cache()
    if row["status"] == "ok":
        print(f"{part:<5} {mode:<13} {name:<9} S={out_size:<3} T={frames:<2} q={str(row['q_chunk']):<5} "
              f"{row['median_ms']:9.2f} ms (IQR {row['iqr_ms']:.2f})  {row['per_frame_ms']:8.2f} ms/frame  "
              f"peak {row['peak_total_mib']:8.1f} MiB  extra {row['peak_extra_mib']:8.1f} MiB")
    else:
        print(f"{part:<5} {mode:<13} {name:<9} S={out_size:<3} T={frames:<2} q={str(row['q_chunk']):<5} OOM")
    return row

In [ ]:
# ── Cell 5 · Main row (both modes) + sweep (fast) ──────────────────────────
rows = []
t_start = time.perf_counter()
set_mode("fast")
for name in UPSAMPLERS:
    rows.append(run_row("main", "fast", name, **MAIN))
if RUN_SWEEP:
    for out_size in OUT_SIZES:                             # small → large, so OOMs come last
        for frames in FRAMES:
            for name in UPSAMPLERS:
                for q in (Q_CHUNKS if name == "anyup3d" else [None]):
                    rows.append(run_row("sweep", "fast", name, out_size, frames, q))
set_mode("deterministic")
for name in UPSAMPLERS:
    rows.append(run_row("main", "deterministic", name, **MAIN))
set_mode("fast")
UPS["anyup3d"].q_chunk_size = MAIN["q_chunk"]              # back to the main value (recorded in the config meta)
print(f"\n{len(rows)} rows ({sum(r['status'] == 'OOM' for r in rows)} OOM) in {time.perf_counter() - t_start:.0f} s")

ok = [r for r in rows if r["status"] == "ok"]
check("output shapes", all(tuple(r["out_shape"]) == (r["frames"], FEAT_DIM, r["out_size"], r["out_size"]) for r in ok),
      f"{len(ok)} ok rows are (T, {FEAT_DIM}, S, S)")
bil = [r for r in ok if r["upsampler"] == "bilinear"]
rel = [abs(r["peak_extra_mib"] - r["frames"] * FEAT_DIM * r["out_size"] ** 2 * 4 / 2**20)
       / (r["frames"] * FEAT_DIM * r["out_size"] ** 2 * 4 / 2**20) for r in bil]
check("bilinear extra memory ≈ output tensor", max(rel) < 0.10, f"max relative gap {max(rel):.3f} over {len(bil)} rows (tol 0.10)")

In [ ]:
# ── Cell 6 · Summary (AnyUp3D / AnyUp2D) + save ────────────────────────────
def main_row(mode, name):
    return next(r for r in rows if r["part"] == "main" and r["mode"] == mode and r["upsampler"] == name)

summary = {"params": PARAMS, "main": MAIN, "ratios_vs_anyup2d": {}, "thesis": THESIS, "checks": CHECKS}
p3, p2 = PARAMS["anyup3d"]["params"], PARAMS["anyup2d"]["params"]
print(f"params: anyup3d {p3:,} vs anyup2d {p2:,} → +{100 * (p3 / p2 - 1):.1f} %   (thesis +{THESIS['params_increase_pct']} %)")
for mode in ("fast", "deterministic"):
    a3, a2 = main_row(mode, "anyup3d"), main_row(mode, "anyup2d")
    if a3["status"] != "ok" or a2["status"] != "ok":
        print(f"{mode}: main row OOM ({a3['status']} / {a2['status']})")
        continue
    r = dict(params_increase_pct=100 * (p3 / p2 - 1), time_ratio=a3["median_ms"] / a2["median_ms"],
             memory_ratio_extra=a3["peak_extra_mib"] / a2["peak_extra_mib"],
             memory_ratio_total=a3["peak_total_mib"] / a2["peak_total_mib"])
    summary["ratios_vs_anyup2d"][mode] = r
    print(f"{mode:<13}: time ×{r['time_ratio']:.2f} (thesis ≈ ×{THESIS['time_ratio']})  "
          f"extra memory ×{r['memory_ratio_extra']:.2f}, total ×{r['memory_ratio_total']:.2f} (thesis ≈ ×{THESIS['memory_ratio']})")

print(f"\n{'mode':<13} {'upsampler':<9} {'ms':>9} {'ms/frame':>9} {'peak MiB':>9} {'extra MiB':>9}")
for mode in ("fast", "deterministic"):
    for name in UPSAMPLERS:
        r = main_row(mode, name)
        if r["status"] == "ok":
            print(f"{mode:<13} {name:<9} {r['median_ms']:>9.2f} {r['per_frame_ms']:>9.2f} {r['peak_total_mib']:>9.1f} {r['peak_extra_mib']:>9.1f}")

config = dict(gpu_profile=GPU, gpu_name=GPU_NAME, run_name=RUN_NAME, upsamplers=UPSAMPLERS, warmup=WARMUP,
              repeats=REPEATS, seed=SEED, main=MAIN, run_sweep=RUN_SWEEP, out_sizes=OUT_SIZES, frames=FRAMES,
              q_chunks=Q_CHUNKS, feat_dim=FEAT_DIM, feat_grid=FEAT_GRID, inputs="synthetic randn, fixed seed",
              modes=["fast (deterministic=False)", "deterministic (main row only)"],
              anyup3d_meta=UPS["anyup3d"]._extra_meta(), anyup2d_meta=UPS["anyup2d"]._extra_meta(),
              repo_branch=REPO_BRANCH)
csv_path, json_path = metrics.save_results(OUT_DIR, rows, summary, config, name="exp0")
n_fail = sum(not c["passed"] for c in CHECKS)
print(f"\n{len(CHECKS)} checks, {n_fail} failed\nsaved {csv_path}\n      {json_path}")

In [ ]:
# ── Cell 7 · Optional: per-frame time vs output size ───────────────────────
if RUN_VISUALS:
    fig, axes = plt.subplots(1, len(FRAMES), figsize=(5 * len(FRAMES), 4), squeeze=False, sharey=True)
    for ax, frames in zip(axes[0], FRAMES):
        for name in UPSAMPLERS:
            for q in (Q_CHUNKS if name == "anyup3d" else [None]):
                pts = [(r["out_size"], r["per_frame_ms"]) for r in rows if r["part"] == "sweep" and r["status"] == "ok"
                       and r["upsampler"] == name and r["frames"] == frames and r["q_chunk"] == q]
                if pts:
                    ax.plot(*zip(*pts), marker="o", label=name if name != "anyup3d" else f"anyup3d q={q}")
        ax.set_title(f"{frames} feature frames"); ax.set_xlabel("output size S"); ax.set_yscale("log")
    axes[0][0].set_ylabel("ms per frame (median)"); axes[0][-1].legend(fontsize=8)
    plt.tight_layout()
    fig_path = f"{OUT_DIR}/exp0_time_vs_size.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()
    print("saved", fig_path)
else:
    print("RUN_VISUALS=False — skipped")